# Flyweight Design Pattern 

explained using the classic Forest (Millions of Trees) example.

#### The Concept

The Flyweight Pattern is used to **save memory** when you need to create a huge number of similar objects. Instead of storing the same data (like a 5MB texture or color) in every single object 1 million times, you store it once (Intrinsic State) and share it. The unique data (like X, Y coordinates) is stored in the individual objects (Extrinsic State).

**Analogy**: A Video Game.
- **Without Flyweight**: Loading 1 million trees consumes 16GB RAM because every tree loads its own texture.
- **With Flyweight**: You load the texture once. All 1 million trees just point to that one texture.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we explicitly separate the **Flyweight** (the shared type) from the **Context** (the tree position), and we use a **Factory** to ensure we don't create duplicate Flyweights.

#### THE FLYWEIGHT (Shared / Intrinsic State)

In [1]:
class TreeType:
    """
    Stores data that is SHARED among millions of trees.
    (Texture, Color, Model).
    """
    def __init__(self, name: str, color: str, texture: str):
        self.name = name
        self.color = color
        self.texture = texture

    def draw(self, x: int, y: int):
        # We receive the unique (extrinsic) state as arguments
        print(f"Drawing {self.name} ({self.color}) at coordinates [{x}, {y}]")

#### THE FACTORY (Manages the Cache)

In [2]:
from typing import Dict

class TreeFactory:
    _tree_types: Dict[str, TreeType] = {}

    @staticmethod
    def get_tree_type(name: str, color: str, texture: str) -> TreeType:
        key = f"{name}_{color}_{texture}"
        
        # If we already made this type, return it. Don't make a new one.
        if key not in TreeFactory._tree_types:
            print(f"Factory: Creating new TreeType '{name}'")
            TreeFactory._tree_types[key] = TreeType(name, color, texture)
        else:
            # Reusing existing object!
            print(f"Factory: Reusing existing '{name}'")
            
        return TreeFactory._tree_types[key]

#### THE CONTEXT (Unique / Extrinsic State)

In [3]:
class Tree:
    """
    This is the object we create millions of.
    It is very light because it only holds coordinates and
    a REFERENCE to the heavy TreeType.
    """
    def __init__(self, x: int, y: int, type: TreeType):
        self.x = x
        self.y = y
        self.type = type

    def render(self):
        # Delegate the drawing to the flyweight
        self.type.draw(self.x, self.y)

#### CLIENT CODE

In [4]:
def main():
    factory = TreeFactory()
    forest = []

    # 1. Plant a Pine Tree
    type_pine = factory.get_tree_type("Pine", "Green", "Rough")
    tree1 = Tree(10, 20, type_pine)
    forest.append(tree1)

    # 2. Plant another Pine Tree (Should reuse the object)
    type_pine2 = factory.get_tree_type("Pine", "Green", "Rough")
    tree2 = Tree(50, 60, type_pine2)
    forest.append(tree2)

    # 3. Plant an Oak Tree (New object)
    type_oak = factory.get_tree_type("Oak", "Brown", "Smooth")
    tree3 = Tree(100, 200, type_oak)
    forest.append(tree3)

    print("\n--- Rendering Forest ---")
    for tree in forest:
        tree.render()

    print(f"\nTotal Tree objects: {len(forest)}")
    print(f"Total TreeTypes in memory: {len(TreeFactory._tree_types)}") 
    # Result: 3 Trees, but only 2 Types in memory.

if __name__ == "__main__":
    main()

Factory: Creating new TreeType 'Pine'
Factory: Reusing existing 'Pine'
Factory: Creating new TreeType 'Oak'

--- Rendering Forest ---
Drawing Pine (Green) at coordinates [10, 20]
Drawing Pine (Green) at coordinates [50, 60]
Drawing Oak (Brown) at coordinates [100, 200]

Total Tree objects: 3
Total TreeTypes in memory: 2


## The Pythonic Way

In Python, we can perform two specific optimizations:
- `__new__` Caching: Instead of an external Factory class, the class can act as its own factory using `cls._cache`. This makes the usage cleaner `(TreeType(...)` automatically returns a cached instance if available).
- `__slots__`: This is a crucial Python memory optimization. By default, Python objects use a dictionary `__dict__` to store attributes, which uses a lot of RAM. Using `__slots__` removes this dictionary and saves `~40-50%` memory per object, which is vital for Flyweight scenarios.

#### THE PYTHONIC FLYWEIGHT (Auto-Caching)

In [5]:
import sys

class TreeModel:
    """
    The Heavy Object.
    Uses __new__ to ensure we never create duplicate models.
    """
    _cache = {}

    def __new__(cls, name: str, color: str):
        # Create a unique key for the state
        key = (name, color)
        
        # If exists, return it immediately
        if key in cls._cache:
            return cls._cache[key]
        
        # Otherwise, create new and store it
        instance = super().__new__(cls)
        cls._cache[key] = instance
        return instance

    def __init__(self, name: str, color: str):
        # Standard init (runs every time, but harmless for simple assignment)
        self.name = name
        self.color = color

#### THE LIGHTWEIGHT CONTEXT (__slots__)

In [6]:
class Tree:
    """
    The Light Object.
    Using __slots__ prevents the creation of __dict__ for every tree,
    saving massive amounts of RAM.
    """
    __slots__ = ['x', 'y', 'model']

    def __init__(self, x, y, model):
        self.x = x
        self.y = y
        self.model = model

    def render(self):
        # We access the shared model data
        return f"Tree at {self.x},{self.y} | Model: {self.model.name}"

#### CLIENT CODE

In [7]:
def main():
    forest = []
    
    print("--- Planting 100,000 Trees ---")
    
    # We only have 2 types of trees: Pine and Oak
    model_pine = TreeModel("Pine", "Green")
    model_oak = TreeModel("Oak", "Orange")

    # Let's verify they are distinct
    print(f"Is Pine same object as Oak? {model_pine is model_oak}") # False

    # Create another Pine - notice we don't need a Factory class
    model_pine_2 = TreeModel("Pine", "Green")
    print(f"Is Pine 2 same object as Pine 1? {model_pine is model_pine_2}") # True!

    # Simulate planting massive forest
    for i in range(100000):
        # Alternating between Pine and Oak
        model = model_pine if i % 2 == 0 else model_oak
        tree = Tree(i, i, model)
        forest.append(tree)

    print(f"Planted {len(forest)} trees.")
    print(f"Actual distinct 'Model' objects in memory: {len(TreeModel._cache)}") # 2
    
    # Memory Check (Conceptual)
    # Without Flyweight: 100,000 strings "Pine", 100,000 strings "Green"
    # With Flyweight: Only 2 strings "Pine", 2 strings "Green"

if __name__ == "__main__":
    main()

--- Planting 100,000 Trees ---
Is Pine same object as Oak? False
Is Pine 2 same object as Pine 1? True
Planted 100000 trees.
Actual distinct 'Model' objects in memory: 2


#### Summary of Differences

| Feature          | Classic OOP                                                        | Pythonic                                                                                  |
|------------------|--------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| **Caching Logic** | Separate `TreeFactory` class checking a `HashMap`.                 | `__new__` method inside the class handles caching transparently.                          |
| **Object Overhead** | Standard Classes.                                                  | `__slots__` used to strip away overhead for millions of tiny objects.                     |
| **Usage**        | `factory.get_tree(...)`   